In [154]:
import re
import pandas as pd
from IPython.display import display, HTML, Markdown
class FullMoonData:
    def __init__(self ) :
        fn = '/Users/sunder/projects/cahc/cahc-utils/datasets/astropy-fm-bce-2500.tsv'
        zdf = pd.read_csv(fn, sep='\t')

        # moon_part_names = sorted(zdf.moon_न.unique().tolist())
        zdf = zdf.dropna().assign (
            date = lambda x: x.fits.apply(lambda z: re.sub("T.*","",z)),
            yr =  lambda x: x.yr.apply(lambda z: int(z-1) if z < 0 else int(z)),
            rtu = lambda x: x.season.apply(lambda s: ['vasantā', 'grīṣma', 'varṣā', 'śarad', 'hemanta', 'śiśira'][s]),
            fm = 1,
            century = lambda x: x.yr.apply(lambda z: int(100*(z//100))).astype(int),
            # century = lambda x: pd.cut(x.yr, bins=range(-2100, -1300, 100)), #, labels=range(-2500, 2500, 100)), 
            # century = lambda x: pd.cut(x.yr, bins=range(-2500, 2500, 100)), #, labels=range(-2500, 2500, 100)), 
            moon_part_num = lambda x: 1+((x.moon_lon*100)//1333),
        #).where(lambda x: x.yr.apply ( lambda y : -2100 <= int(y ) < -1300)).dropna(
        # ).where(lambda x: x.moon_part_num.apply ( lambda y : 1 <= int(y ) <= 27)).dropna().assign(
        #     moon_part = lambda x: x.moon_part_num.apply(lambda z: moon_part_names[int(z)-1]),
        ).dropna().map(lambda x: int(x) if isinstance(x,float) else x).dropna().reset_index(drop=True)
        self.fm_df = zdf

    def six_monthly_fm(self, init_row=0):
        def concat(srs):
            try :
                return pd.Series([ x*(10**(len(srs)-ix-1)) for ix, x in enumerate(srs) ]).sum()
            except:
                return 111111

        zdf = self.fm_df.copy()
        # select every sixth row starting from init_row 
        zdf = zdf.iloc[init_row::6].reset_index(drop=True)
        zdf = zdf.assign(
            six_fm_diff = zdf.jd - zdf.jd.shift(1),
        ).dropna()
        zdf = zdf.assign(
            smud=lambda x: (x.six_fm_diff % 10).apply(lambda y: int(str(y)[0]))
        ).assign(
            six_adjacent_smuds=lambda x: x.smud.rolling(6).apply( concat )#.fillna(222222).apply(int)
        ).assign(
            reps = lambda x: x.six_adjacent_smuds.apply(lambda z: re.match(r".*(.)\1{5,}.*", str(z)) is not None)
        ).dropna()#.apply(int)
        return zdf


    def iks_2003_prep (self ):
        zdf = self.fm_df.copy()
        zdf =  zdf.where(lambda x: x.yr.apply ( lambda y : -2100 <= int(y ) < -1300)).dropna()
        cols = [# 'iter', 
                'date', 'jd','stel_jd', 
                #'yr',
                #'sun_ra', 'sun_dec', 
                'sun_lon', #'sun_lat',
            #'moon_ra', 'moon_dec', 
            'moon_lon', 'moon_lat', #'phase', 'jd_diff',
            #'sun_lat_lon', 'moon_lat_lon', 
            #'yr', 'stel_jd', 
            'sun_naks', 'moon_naks', 'moon_part'
            #'sun_न', 'moon_न', 'moon_phase', 
            'season', 'rtu'
            'eqnx_1_tol', 'eqnx_2_tol',
            #'eqnx_3_tol', 'eqnx_4_tol'
            ]
        zdf.index = zdf.index + 1
        # zdf[cols].yr.value_counts().sort_index()#.plot(kind='bar', figsize=(20,5), title="Number of observations per year")

        fm_by_century = zdf[ ['fm' , 'century' , ]].groupby(['century']).count()
        ax = fm_by_century.plot(kind='bar', figsize=(15.7,4), fontsize=13, rot=0, color='brown', alpha=0.5, legend=None)
        # annotate each bar with values
        for p in ax.patches: ax.annotate(f"{p.get_height():,}", (p.get_x() +.15, p.get_height() * (1-.1)), fontsize=13)

        ax.set_title("Number of Full Moons per century", fontsize=17)
        ax.grid(ls=':', color='gray', alpha=0.5)
        ax.set_xlabel("", fontsize=2)
        # ax.set_ylim(1230, 1238)
        slice = zdf[ ['fm' , 'century' , 'moon_न' , 'sun_lon', 'eqnx_1_tol']]
        tol = 2

        efms = slice [ slice.eqnx_1_tol]
        ax = efms[ ['fm' , 'century' , ]].groupby(['century']).count().plot(kind='bar', figsize=(16,4), fontsize=13, rot=0, color='green', alpha=0.5, legend=None)# ( ax=ax, logy=True)
        for p in ax.patches: ax.annotate(f"{p.get_height():,}", (p.get_x() +.15, p.get_height() * (1-.1)), fontsize=13)
        ax.set_title("Number of Equinoctial Full Moons per century ", fontsize=17)
        ax.grid(ls=':', color='gray', alpha=0.5)
        ax.set_xlabel("", fontsize=2)
        # ax.set_yticks([1, 10, 20,30,40,50,100,200,300,400,500,1000,2000,3000,4000,5000])

        efms_kv = efms[ efms.moon_न.apply(lambda z: 'Kri' in z or 'Vis' in z)]

        for ix, slicex in enumerate ( [efms_kv]) :
            pvt = slicex.pivot_table(index=['century'], columns=['moon_न'], values=['fm'], aggfunc='sum')
            # pvt.iloc[3,0] =3; pvt.iloc[3,1] =6    
            # display(pvt.index)
            # display(pvt.fillna(0).style.format(precision=0).background_gradient(cmap='Blues', axis=None))
            colors = [ 'red', 'blue',]
            ax = pvt.plot(kind='bar', stacked=True, figsize=(16,4)
                # , title="Number of FullMoon per century by Visible Kṛttikā and Viśākhā Nakṣatras"
                , fontsize=13, rot=0
                , color= colors if ix==0 else colors[::-1]
                )
            for p in ax.patches: ax.annotate(f"{int(p.get_height()):,}", (p.get_x() +.15, p.get_height() * (1-.1)), fontsize=13, color='white')
            ax.set_title ("Number of Equinoctial Full Moons per century near Visible Kṛttikā and Viśākhā Nakṣatras", fontsize=17)
            ax.grid(ls=':', color='gray', alpha=0.5)
            ax.set_xlabel("", fontsize=2)
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(reversed(handles), reversed(labels), title=' ', fontsize=12)
            # draw a box around the 3rd century
            ax.axvspan(.5, 4.5, facecolor='yellow', alpha=0.25)

FM = FullMoonData()
df_acc = []
for i in range(6):
    sdf = FM.six_monthly_fm(init_row=i)
    idxs = sdf[sdf.reps].index.to_list()

    for ix, idx in enumerate(sorted(set(idxs))):
        _df = sdf.loc[idx-5:idx]
        if len(_df) == 6:
            df_acc.append(_df)
            # display(ix, _df[['fits', 'jd', 'six_adjacent_smuds', 'moon_naks', 'reps']].style)

xdf = pd.concat([ x for x in df_acc if len(x)==6]).sort_values('jd').assign(
    new_block=lambda x: x.reps.shift(1),
)

xdf = xdf[['fits', 'jd', 'six_adjacent_smuds', 'moon_naks', 'new_block']]
# display(HTML("<h2>Full Moon smuds</h2>"))

In [159]:
# Reset the index before applying styling
xdf = xdf.reset_index(drop=True)

# Display the dataframe with styled rows
def highlight_new_blocks(row):
    return ['background-color: lightcyan' if row.new_block else '' for _ in row]

# Fill NaN values in new_block column to avoid styling issues
xdf['new_block'] = xdf['new_block'].fillna(False)
xdf.loc[0, 'new_block'] = True  # Mark the first row as a new block
xdf = xdf.assign(block_num=lambda x: x.new_block.cumsum()).set_index('block_num', append=not True, drop=True)
# Display the styled dataframe with highlighted new blocks , and hide the new_block column using the style.apply method

display(
    xdf.reset_index().style.apply(highlight_new_blocks, axis=1).hide(axis='columns', subset=['new_block']).format(precision=0)
)

# save the dataframe to a styled HTML file
xdf.reset_index().style.apply(highlight_new_blocks, axis=1).hide(axis='columns', subset=['new_block']).format(
    precision=0
).to_html('../../fm-smuds~.html', index=False, justify='left', classes='table table-striped table-hover')

,block_num,fits,jd,six_adjacent_smuds,moon_naks
0,1,-02436-07-07T13:17:09.781,831517,868787,N24-Sha/λ Aqr
1,1,-02436-12-31T13:56:30.372,831694,687877,N11-PPal/δ Leo
2,1,-02435-06-27T03:51:06.977,831871,878777,N24-Sha/λ Aqr
3,1,-02435-12-20T20:48:21.982,832048,787777,N10-Mag/α Leo
4,1,-02434-06-16T10:27:26.006,832225,877777,N23-Dha/γ2 Del
5,1,-02434-12-10T10:17:57.919,832402,777777,N10-Mag/ε Leo
6,2,-02391-08-18T14:09:29.753,847995,869687,N27-Rev/ε Psc
7,2,-02390-02-11T15:39:54.124,848172,696877,N14-Chi/α Vir
8,2,-02390-08-08T04:42:58.504,848349,968777,N26-UBha/α And
9,2,-02389-01-31T21:40:55.048,848526,687777,N13-Has/ε Crv
